# Autonomous Visual AI Agent for Precision Agriculture: SAGE Tomato Vision Model Training
**Model:** Qwen2.5-VL-3B-Instruct (Loaded from Kaggle Input)  
**Fine-Tuning:** 4-bit NF4 QLoRA  
**Dataset:** `IDK-Chaitanya/SAGE5class` (Selective download for Tomato)  
**Target Duration:** ~3–4 hours on Kaggle Cloud GPU (T4 / P100 / L4 / A100)

### Step 1: Environment & GPU Inspection
Check available GPU, CUDA capability, and VRAM.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

### Step 2: Install Pinned Dependencies
Ensure compatible versions of Transformers, PEFT, BitsAndBytes, and Qwen-VL utilities are present.

In [ ]:
%cd /kaggle/working
!pip install -q -U accelerate bitsandbytes peft transformers qwen-vl-utils pyyaml seaborn

### Step 3: Clone or Update GitHub Repository
Replace `YOUR_GITHUB_USERNAME` with your actual repository URL.

In [ ]:
import os
import sys

# Set your repository URL and branch here
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/sage-qwen-trainer.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/sage-qwen-trainer"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

# Add repository to Python module path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}
!git rev-parse HEAD

### Step 4: Hugging Face Token Configuration (Optional/Secure)
If your dataset is private, retrieve token from Kaggle Secrets.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face token loaded securely from Kaggle Secrets.")
except Exception:
    print("No Kaggle Secret 'HF_TOKEN' detected. Continuing with public access.")

### Step 5: Locate Qwen2.5-VL Model in Kaggle Input
Scans `/kaggle/input/` for attached Qwen2.5-VL-3B-Instruct model weights.

In [ ]:
from src.model_loader import discover_qwen_model_path

discovered_model = discover_qwen_model_path("/kaggle/input")
if discovered_model:
    os.environ["QWEN_MODEL_PATH"] = discovered_model
    print(f"Selected Base Model Directory: {discovered_model}")
else:
    print("No local Qwen model found in /kaggle/input. Model loader will attempt online Hugging Face download.")

### Step 6: Validate Dataset (Selective Tomato Download)
Downloads only metadata and Tomato images, validating splits and checking for data leakage.

In [ ]:
!python scripts/validate_dataset.py --config configs/config.yaml --crop Tomato

### Step 7: Multimodal Forward-Pass Smoke Test
Executes a forward and backward pass with synthetic data to verify finite loss and gradient flow to LoRA adapter.

In [ ]:
!python scripts/smoke_test.py --config configs/config.yaml

### Step 8: Execute QLoRA Training Pipeline
Runs full training with live progress bar, batch/epoch tracking, validation checkpoints, and held-out test evaluation.

In [ ]:
from src.train import run_training_pipeline

# Train with settings from configs/config.yaml
output_run_dir = run_training_pipeline(config_path="configs/config.yaml")
print(f"\nTraining finished! Run artifacts are stored at: {output_run_dir}")

### Step 9: View Generated Reports & Confusion Matrix

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

run_path = Path(output_run_dir)
summary_file = run_path / "run_summary.json"
if summary_file.exists():
    with open(summary_file, "r") as f:
        summary = json.load(f)
    print("\n--- RUN SUMMARY ---")
    print(f"Crop: {summary.get('crop')}")
    print(f"Total Execution: {summary.get('total_execution_seconds', 0) / 60:.1f} minutes")
    test_m = summary.get("test_metrics", {})
    print(f"Test Accuracy: {test_m.get('accuracy', 0):.4f}")
    print(f"Test Macro F1: {test_m.get('f1_macro', 0):.4f}")

cm_plot = run_path / "confusion_matrix_test.png"
if cm_plot.exists():
    display(Image(str(cm_plot)))

### Step 10: Package Artifacts for Download
Compresses the final LoRA adapter, training logs, and metrics into a lightweight archive.

In [ ]:
!tar -czvf /kaggle/working/sage_tomato_adapter_and_reports.tar.gz -C {output_run_dir} adapter batch_loss.csv epoch_metrics.csv run_summary.json model_card.md
print("Package ready for download: /kaggle/working/sage_tomato_adapter_and_reports.tar.gz")